In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
DATA_DIR = Path("../data")

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
articles = pd.read_csv(
    RAW_DIR / "articles.csv"
)

print(articles.shape)

articles.head(2)

(105542, 25)


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.


In [4]:
customers = pd.read_csv(
    RAW_DIR / "customers.csv"
)

print(customers.shape)

customers.head()

(1371980, 7)


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [5]:
transactions = pd.read_csv(
    RAW_DIR / "transactions_train.csv",
    usecols=[
        "t_dat",
        "customer_id",
        "article_id",
        "price"
    ]
)

print(transactions.shape)

transactions.head()

(31788324, 4)


,t_dat,customer_id,article_id,price
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932


In [6]:
articles["article_id"] = (
    articles["article_id"]
    .astype("int32")
)

transactions["article_id"] = (
    transactions["article_id"]
    .astype("int32")
)

transactions["price"] = (
    transactions["price"]
    .astype("float32")
)

transactions["t_dat"] = pd.to_datetime(
    transactions["t_dat"]
)

In [7]:
user_activity = (
    transactions
    .groupby("customer_id")
    .size()
    .rename("num_purchases")
    .reset_index()
)

user_activity.head()

,customer_id,num_purchases
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,21
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,86
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,18
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,2
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,13


In [8]:
N_USERS = 100_000

In [9]:
top_users = (
    user_activity
    .sort_values(
        "num_purchases",
        ascending=False
    )
    .head(N_USERS)
)

top_users.shape

(100000, 2)

In [10]:
transactions = transactions[
    transactions["customer_id"].isin(
        top_users["customer_id"]
    )
].copy()

transactions.shape

(13025149, 4)

In [11]:
valid_articles = (
    transactions["article_id"]
    .unique()
)

articles = articles[
    articles["article_id"]
    .isin(valid_articles)
].copy()

articles.shape

(99604, 25)

In [12]:
customers = customers[
    customers["customer_id"]
    .isin(top_users["customer_id"])
].copy()

customers.shape

(100000, 7)

In [13]:
customers["age"] = customers["age"].fillna(
    customers["age"].median()
)

customers["fashion_news_frequency"] = (
    customers["fashion_news_frequency"]
    .fillna("NONE")
)

In [14]:
np.random.seed(42)

business_features = articles[
    ["article_id"]
].copy()

business_features["margin_pct"] = np.random.uniform(
    0.10,
    0.70,
    len(business_features)
)

business_features["inventory"] = np.random.randint(
    50,
    500,
    len(business_features)
)

business_features["promotion_priority"] = np.random.randint(
    1,
    6,
    len(business_features)
)

business_features.head()

,article_id,margin_pct,inventory,promotion_priority
0,108775015,0.324724,470,5
1,108775044,0.670429,63,1
2,108775051,0.539196,372,3
3,110065001,0.459195,358,3
4,110065002,0.193611,304,2


In [15]:
interactions = (
    transactions[
        ["customer_id", "article_id"]
    ]
    .drop_duplicates()
    .assign(interaction=1)
)

interactions.head()

,customer_id,article_id,interaction
2,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,1
3,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,1
4,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,1
5,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687001,1
6,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221001,1


In [16]:
print(
    f"Users: {interactions['customer_id'].nunique():,}"
)

print(
    f"Products: {interactions['article_id'].nunique():,}"
)

print(
    f"Interactions: {len(interactions):,}"
)

Users: 100,000
Products: 99,604
Interactions: 10,725,535


In [17]:
articles.to_parquet(
    PROCESSED_DIR / "articles.parquet",
    index=False
)

customers.to_parquet(
    PROCESSED_DIR / "customers.parquet",
    index=False
)

transactions.to_parquet(
    PROCESSED_DIR / "transactions.parquet",
    index=False
)

business_features.to_parquet(
    PROCESSED_DIR / "business_features.parquet",
    index=False
)

interactions.to_parquet(
    PROCESSED_DIR / "interactions.parquet",
    index=False
)